# 1. Text-Only Candidate–History Attention

A text-only reranker that asks: *which of the user's earlier clicks are relevant to this
candidate?* No tabular features are used.

- **Article encoder**: frozen 64-dimensional text embeddings shared by candidates and history
- **Candidate–history attention**: the candidate is the query and the clicked articles are
  keys and values, so the history summary changes with every candidate
- **Scoring head**: the candidate vector and its history context go through a small MLP

The ablation compares one attention head with four; validation nDCG@10 selects the final
model and only that model is tested.

## 1.1 Load Packages

In [ ]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = sorted((RUN / f"ranking_features_{split}").glob("*.parquet"))[0]
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Experiment Configuration

In [ ]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

TRAIN_ROWS = 250_000
HISTORY_CAP = 50
EPOCHS = 5
BATCH_SIZE = 2048
HEAD_OPTIONS = (1, 4)
SELECTION_METRIC = "ndcg_at_10"

# 3. Prepare Training Data

## 3.1 Sample Labelled Training Candidates

`sample_train_rows` draws a reproducible, uniform sample of training candidates. The
number of rows taken from each partition follows a hypergeometric draw, so the whole
training split never has to be loaded at once.

In [ ]:
def read_labelled_part(part, columns):
    """Join one training feature partition to its click labels."""
    features = pq.read_table(part, columns=["impression_uid", "news_code", *columns]).to_pandas()
    labels = pq.read_table(
        RUN / "labels_train" / part.name,
        columns=["impression_uid", "news_code", "clicked"],
    ).to_pandas()
    return features.merge(labels, on=["impression_uid", "news_code"], validate="one_to_one")


def sample_train_rows(limit, columns):
    """Sample `limit` labelled training candidates uniformly across all partitions."""
    parts = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    sizes = [pq.ParquetFile(part).metadata.num_rows for part in parts]
    rng = np.random.default_rng(SEED)
    remaining_population = sum(sizes)
    remaining_take = min(limit, remaining_population)

    samples = []
    for part, size in zip(parts, sizes):
        take = int(rng.hypergeometric(size, remaining_population - size, remaining_take))
        if take:
            frame = read_labelled_part(part, columns)
            samples.append(frame.iloc[rng.choice(size, take, replace=False)])
        remaining_population -= size
        remaining_take -= take
    return pd.concat(samples, ignore_index=True)

## 3.2 Article Vectors and Causal Click Histories

Every article is represented by its frozen 64-dimensional text embedding. A user's
history is the list of articles they clicked **strictly before** the impression, saved
by feature engineering; at most the newest `HISTORY_CAP` clicks are used.

In [ ]:
# Map each news_code to its row in the frozen 64-dimensional text embedding matrix.
# Row 0 is reserved for padding.
article_index = pl.read_parquet(REPRESENTATIONS / "article_index.parquet", columns=["news_code", "article_row"])
news_to_row = dict(zip(article_index["news_code"].to_list(), article_index["article_row"].to_list()))
text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
EMBEDDING_DIM = text_vectors.shape[1]
print(f"Article embeddings: {text_vectors.shape[0]:,} rows x {EMBEDDING_DIM} dimensions")

In [ ]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = (
        pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet")
        .select("impression_uid", "full_history_rows")
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

In [ ]:
def encode_pairs(frame, profiles):
    """Return the padded click history and the candidate article row for every candidate.

    Histories keep the newest HISTORY_CAP clicks and are left-padded with row 0.
    """
    histories = np.zeros((len(frame), HISTORY_CAP), dtype=np.int64)
    candidates = np.empty(len(frame), dtype=np.int64)
    for index, (uid, news_code) in enumerate(zip(frame["impression_uid"], frame["news_code"])):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
        candidates[index] = news_to_row[news_code]
    return histories, candidates

In [ ]:
train = sample_train_rows(TRAIN_ROWS, columns=[])
train_profiles = load_profiles("train", train["impression_uid"].unique())
train_history, train_candidate = encode_pairs(train, train_profiles)
train_labels = train["clicked"].to_numpy(np.float32)
print(f"Sampled {len(train):,} candidates from {len(train_profiles):,} causal histories")

# 4. Model: `CandidateHistoryAttention`

```text
Candidate text ─→ query ─┐
History text ──→ keys / values → multi-head attention → history context
                                         ↓
            [candidate vector ; history context] → Dense 128 → ReLU → logit
```

Padding positions are masked. `nn.MultiheadAttention` cannot attend over a fully masked
sequence, so users with no history temporarily expose one padding slot and then receive
a zero context vector.

In [ ]:
class CandidateHistoryAttention(nn.Module):
    """Candidate-conditioned multi-head attention over a user's causal text history."""

    def __init__(self, vectors, num_heads):
        super().__init__()
        self.articles = nn.Embedding.from_pretrained(
            torch.as_tensor(np.asarray(vectors), dtype=torch.float32),
            freeze=True,
            padding_idx=0,
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=EMBEDDING_DIM,
            num_heads=num_heads,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(2 * EMBEDDING_DIM, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, history_rows, candidate_rows):
        history = self.articles(history_rows)          # [batch, history, 64]
        candidate = self.articles(candidate_rows)      # [batch, 64]

        has_history = history_rows.ne(0).any(dim=1)
        padding_mask = history_rows.eq(0).clone()
        padding_mask[~has_history, 0] = False          # keep one slot open for empty histories

        context, _ = self.attention(
            query=candidate.unsqueeze(1),
            key=history,
            value=history,
            key_padding_mask=padding_mask,
            need_weights=False,
        )
        context = context.squeeze(1) * has_history.unsqueeze(1)   # zero context for cold users
        return self.head(torch.cat([candidate, context], dim=1)).squeeze(-1)

# 5. Training

Weighted binary cross-entropy, with the positive-class weight taken from the training
click rate.

In [ ]:
def train_attention(num_heads):
    """Train one candidate-attention model with weighted BCE."""
    torch.manual_seed(SEED)
    model = CandidateHistoryAttention(text_vectors, num_heads).to(DEVICE)
    click_rate = train_labels.mean()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - click_rate) / click_rate], device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(train_history),
            torch.from_numpy(train_candidate),
            torch.from_numpy(train_labels),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for histories, candidates, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(histories.to(DEVICE), candidates.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        print(f"  epoch {epoch + 1}/{EPOCHS}: train BCE={total_loss / len(train_labels):.5f}")
    return model

In [ ]:
def predict_batches(model, *inputs):
    """Score aligned NumPy inputs in mini-batches and return click probabilities."""
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(inputs[0]), BATCH_SIZE):
            batch = [torch.from_numpy(array[start:start + BATCH_SIZE]).to(DEVICE) for array in inputs]
            scores.append(torch.sigmoid(model(*batch)).cpu().numpy())
    return np.concatenate(scores)

In [ ]:
def attention_predictor(model, profiles):
    """Wrap a trained model and a split's histories as a predict(frame) function."""
    def predict(frame):
        histories, candidates = encode_pairs(frame, profiles)
        return predict_batches(model, histories, candidates)
    return predict

# 6. Validation Comparison

Each head count is trained, saved and evaluated on the complete validation slates.

In [ ]:
validation_profiles = load_profiles("validation")
models, validation_results = {}, []
for num_heads in HEAD_OPTIONS:
    name = f"attention_text_h{num_heads}"
    print(f"\n{name}")
    model = train_attention(num_heads)
    models[name] = model

    output_dir = ARTIFACTS / name
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {"state_dict": model.state_dict(), "num_heads": num_heads, "history_cap": HISTORY_CAP,
         "embedding": "text64", "train_rows": len(train), "seed": SEED},
        output_dir / "model.pt",
    )
    result, _ = score_and_evaluate(
        attention_predictor(model, validation_profiles),
        split="validation",
        feature_run=RUN,
        output_dir=output_dir,
        model_name=name,
        results_dir=RESULTS,
    )
    validation_results.append(result)

leaderboard = compare_results(validation_results, sort_by=SELECTION_METRIC)
final_model_name = leaderboard["model"][0]
print(f"Selected using validation {SELECTION_METRIC}: {final_model_name}")
display(leaderboard.select("model", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

# 7. Test Evaluation

Only the validation-selected head count is scored on test.

In [ ]:
test_result, _ = score_and_evaluate(
    attention_predictor(models[final_model_name], load_profiles("test")),
    split="test",
    feature_run=RUN,
    output_dir=ARTIFACTS / final_model_name,
    model_name=final_model_name,
    results_dir=RESULTS,
)
display(test_result.select("model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

# 8. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(example_user, score_file=ARTIFACTS / final_model_name / "validation_scores.parquet", feature_run=RUN, top_n=5)